# Gaussian Processes

Companion notebook for the [Gaussian Processes lesson](https://ml-viz-ruby.vercel.app/courses/bayesian-methods/02-gaussian-processes).

We implement **GP regression from scratch** with an RBF kernel: sample functions from the prior,
condition on a few observations to get the closed-form posterior mean and variance, and watch the
length-scale reshape the fit. Pure NumPy + Matplotlib.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#444', 'axes.labelcolor': '#ccc',
    'xtick.color': '#888', 'ytick.color': '#888',
    'text.color': '#eee', 'grid.color': '#333', 'lines.linewidth': 2,
})
rng = np.random.default_rng(1)

## 1 — The RBF kernel and prior samples

The kernel sets how correlated two outputs are. Sampling from N(0, K) over a grid gives random
*functions* — all smooth, with wiggliness set by the length-scale.

In [ ]:
def rbf_kernel(A, B, ell=1.0, sig=1.0):
    d2 = (A[:, None] - B[None, :]) ** 2
    return sig**2 * np.exp(-d2 / (2 * ell**2))

xs = np.linspace(-5, 5, 120)
K = rbf_kernel(xs, xs, ell=1.0) + 1e-9 * np.eye(len(xs))
L = np.linalg.cholesky(K)
fig, ax = plt.subplots(figsize=(8, 3.5))
for _ in range(5):
    ax.plot(xs, L @ rng.normal(size=len(xs)), alpha=0.8)
ax.set_title('5 functions sampled from the GP prior (RBF, ℓ=1)')
ax.set_xlabel('x'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 2 — The closed-form GP posterior

Condition on observations: μ* = K*ᵀ(K+σ²I)⁻¹y,  Σ* = K** − K*ᵀ(K+σ²I)⁻¹K*. We plot the mean and the
±2σ band — note it pinches at the data and widens away from it.

In [ ]:
def gp_posterior(X, y, Xs, ell=1.0, sig=1.0, noise=0.04):
    K = rbf_kernel(X, X, ell, sig) + noise * np.eye(len(X))
    Ks = rbf_kernel(X, Xs, ell, sig)
    Kss = rbf_kernel(Xs, Xs, ell, sig)
    Kinv = np.linalg.inv(K)
    mu = Ks.T @ Kinv @ y
    cov = Kss - Ks.T @ Kinv @ Ks
    return mu, np.sqrt(np.clip(np.diag(cov), 1e-9, None))

Xo = np.array([-3.0, -1.8, -0.5, 1.2, 2.6])
yo = np.array([-1.2, 0.9, 0.4, -0.8, 1.1])
mu, sd = gp_posterior(Xo, yo, xs, ell=1.0)

fig, ax = plt.subplots(figsize=(8, 4))
ax.fill_between(xs, mu - 2*sd, mu + 2*sd, color='#6366f1', alpha=0.3, label='±2σ posterior')
ax.plot(xs, mu, color='#818cf8', label='posterior mean')
ax.scatter(Xo, yo, color='#2dd4bf', zorder=5, label='observations')
ax.set_title('GP posterior: confident at data, uncertain away from it')
ax.legend(facecolor='#1a1d27', edgecolor='#444'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print('σ at an observation (x=-0.5):', round(gp_posterior(Xo, yo, np.array([-0.5]))[1][0], 3))
print('σ far from data  (x=5.0):   ', round(gp_posterior(Xo, yo, np.array([5.0]))[1][0], 3))

## 3 — The length-scale reshapes the fit

Small ℓ → wiggly, uncertain between points; large ℓ → smooth, confident. There's no single 'right'
value — in practice it's chosen by maximizing the marginal likelihood.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), sharey=True)
for ax, ell in zip(axes, [0.4, 1.0, 2.5]):
    mu, sd = gp_posterior(Xo, yo, xs, ell=ell)
    ax.fill_between(xs, mu-2*sd, mu+2*sd, color='#6366f1', alpha=0.3)
    ax.plot(xs, mu, color='#818cf8')
    ax.scatter(Xo, yo, color='#2dd4bf', zorder=5)
    ax.set_title(f'ℓ = {ell}'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## ✏️ Your turn

**Exercise.** Implement `rbf(a, b, ell, sig)` for two scalars and `posterior_var(X, Xs, ell, sig,
noise)` returning the GP posterior variance at each test point in `Xs`:
diag(K** − K*ᵀ(K+σ²I)⁻¹K*). (The mean needs y; the variance does not — it depends only on *where*
the data is, not its values.)

In [ ]:
def rbf(a, b, ell=1.0, sig=1.0):
    # TODO(you): scalar RBF kernel sig^2 * exp(-(a-b)^2 / (2 ell^2))
    return ...

def posterior_var(X, Xs, ell=1.0, sig=1.0, noise=0.04):
    # TODO(you): return the vector of posterior variances at the points in Xs
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
assert np.isclose(rbf(1.0, 1.0), 1.0)                      # zero distance -> max covariance
assert rbf(0.0, 3.0) < rbf(0.0, 1.0)                       # farther -> less correlated

v = posterior_var(Xo, xs)
near = posterior_var(Xo, np.array([-0.5]))[0]
far  = posterior_var(Xo, np.array([5.0]))[0]
assert far > near                                           # variance is small near an observation, large far away
assert np.all(v >= -1e-6)                                    # variances are non-negative

# Edge case: a single training point. Right on top of it the posterior variance
# shrinks to the noise floor; far away it reverts to the prior variance (sig^2).
v_one_here = posterior_var(np.array([0.0]), np.array([0.0]), noise=0.04)[0]
v_one_far  = posterior_var(np.array([0.0]), np.array([5.0]), noise=0.04)[0]
assert np.isclose(v_one_here, 0.04, atol=0.01)
assert np.isclose(v_one_far, 1.0, atol=0.05)

# Edge case: predicting exactly at one of several training points still shrinks
# variance toward the noise floor, well below the "far" value above.
v_at_train = posterior_var(Xo, np.array([Xo[2]]), noise=0.04)[0]
assert np.isclose(v_at_train, 0.04, atol=0.01)
assert v_at_train < 0.1 * far

# Edge case: a very large length-scale makes the kernel nearly constant, so
# variance stops localizing around the data -- it stays small and roughly flat
# across the whole test range instead of pinching near observations and
# blowing up away from them.
v_large_ell = posterior_var(Xo, xs, ell=1000.0, noise=0.04)
assert np.ptp(v_large_ell) < 0.01                            # nearly flat everywhere
assert v_large_ell.max() < 0.05                              # and small throughout

print(f'✓ kernel and posterior variance correct (near={near:.3f} < far={far:.3f})')

<details>
<summary>Solution</summary>

```python
def rbf(a, b, ell=1.0, sig=1.0):
    return sig**2 * np.exp(-(a - b)**2 / (2 * ell**2))

def posterior_var(X, Xs, ell=1.0, sig=1.0, noise=0.04):
    K = rbf_kernel(X, X, ell, sig) + noise * np.eye(len(X))
    Ks = rbf_kernel(X, Xs, ell, sig)
    Kss = rbf_kernel(Xs, Xs, ell, sig)
    return np.diag(Kss - Ks.T @ np.linalg.inv(K) @ Ks)
```

The variance depends only on the *locations* of the data, not the observed y-values -- which is why
active learning can pick where to sample next (highest variance) before seeing any label there.

The three edge cases above are all instances of the same formula: with one relevant observation,
`diag(K** - K*ᵀ(K+σ²I)⁻¹K*)` reduces to `noise · σ² / (σ² + noise)` -- essentially the noise floor,
whether that "one relevant observation" is the entirety of the training set or just the nearest
point among several. A huge length-scale instead breaks the *locality* of the kernel: every pair of
points looks equally correlated, so the posterior treats the whole test range as "close to the
data" and reports uniformly low variance -- confident, but for the wrong reason.

</details>

## Extra practice -- DML 186: a full `GaussianProcessRegression` class

[Open-Deep-ML problem 186](https://github.com/Open-Deep-ML/DML-OpenProblem/tree/main/questions/186_guassian_mixture_regression)
is filed under a misleading slug ("gaussian mixture regression"), but its
`description.md` and `tests.json` both confirm the actual task: a from-scratch
**Gaussian Process Regression** estimator with a configurable kernel. It
generalizes the `gp_posterior` function above into a stateful,
`scikit-learn`-style class with this exact signature:

`GaussianProcessRegression(kernel='rbf', noise=1e-5, kernel_params=None)`,
with `.fit(X, y)` and `.predict(X_test, return_std=False)`.

(DML's full kernel bank also has Matérn, periodic, and rational-quadratic
variants -- this exercise implements the two used by its test cases, `'rbf'`
and `'linear'`.)

In [ ]:
def rbf_kernel_dml(x, x_prime, sigma=1.0, length_scale=1.0):
    sq_norm = np.linalg.norm(x - x_prime) ** 2
    return sigma**2 * np.exp(-sq_norm / (2 * length_scale**2))

def linear_kernel_dml(x, x_prime, sigma_b=1.0, sigma_v=1.0):
    return sigma_b**2 + sigma_v**2 * np.dot(x, x_prime)

class GaussianProcessRegression:
    def __init__(self, kernel='rbf', noise=1e-5, kernel_params=None):
        self.kernel_name = kernel
        self.noise = noise
        self.kernel_params = kernel_params if kernel_params else {}

    def _select_kernel(self, x1, x2):
        if self.kernel_name == 'rbf':
            return rbf_kernel_dml(x1, x2, **self.kernel_params)
        elif self.kernel_name == 'linear':
            return linear_kernel_dml(x1, x2, **self.kernel_params)
        raise ValueError(f"Unsupported kernel: {self.kernel_name}")

    def _compute_covariance(self, X1, X2):
        X1, X2 = np.atleast_2d(X1), np.atleast_2d(X2)
        K = np.zeros((len(X1), len(X2)))
        for i in range(len(X1)):
            for j in range(len(X2)):
                K[i, j] = self._select_kernel(X1[i], X2[j])
        return K

    def fit(self, X, y):
        # TODO(you): store X, y as float arrays; build K = cov(X, X) + noise * I;
        # solve K @ alpha = y for alpha (use np.linalg.solve, don't invert K directly)
        ...

    def predict(self, X_test, return_std=False):
        # TODO(you): K_s = cov(X_train, X_test); mu = K_s.T @ alpha.
        # If return_std: K_ss = cov(X_test, X_test), cov = K_ss - K_s.T @ solve(K, K_s),
        # return (mu, sqrt(clip(diag(cov), 0, None))); otherwise return mu alone.
        ...

In [ ]:
# This assert cell passes silently when your implementation is correct.

# DML 186 test case: perfectly linear data, linear kernel -> exact interpolation.
gp = GaussianProcessRegression(kernel='linear', kernel_params={'sigma_b': 0.0, 'sigma_v': 1.0}, noise=1e-8)
gp.fit(np.array([[1], [2], [4]]), np.array([3, 5, 9]))       # y = 2x + 1
assert np.isclose(gp.predict(np.array([[3.0]]))[0], 7.0, atol=1e-3)

# DML 186 test case: RBF kernel, mean + std at a point between observations.
gp2 = GaussianProcessRegression(kernel='rbf', kernel_params={'sigma': 1.0, 'length_scale': 1.0}, noise=1e-8)
X5 = np.array([[0], [2.5], [5.0], [7.5], [10.0]])
gp2.fit(X5, np.sin(X5).ravel())
mu, std = gp2.predict(np.array([[1.25]]), return_std=True)
assert np.isclose(mu[0], 0.2814, atol=1e-3)
assert np.isclose(std[0], 0.7734, atol=1e-3)

# Edge case: a single training point. Right on top of it, std collapses to the
# noise floor (sqrt(noise)); far away it reverts to the prior std (sigma).
gp3 = GaussianProcessRegression(kernel='rbf', kernel_params={'sigma': 1.0, 'length_scale': 1.0}, noise=1e-6)
gp3.fit(np.array([[0.0]]), np.array([2.0]))
_, std_here = gp3.predict(np.array([[0.0]]), return_std=True)
_, std_far  = gp3.predict(np.array([[10.0]]), return_std=True)
assert np.isclose(std_here[0], np.sqrt(1e-6), atol=1e-4)
assert np.isclose(std_far[0], 1.0, atol=0.05)

# Edge case: predicting exactly at one of several training points still shrinks
# std to (near) the noise floor, regardless of the other observations around it.
gp4 = GaussianProcessRegression(kernel='rbf', kernel_params={'sigma': 1.0, 'length_scale': 1.0}, noise=1e-6)
gp4.fit(X5, np.sin(X5).ravel())
mu_at, std_at = gp4.predict(np.array([[2.5]]), return_std=True)
assert np.isclose(std_at[0], np.sqrt(1e-6), atol=1e-4)
assert np.isclose(mu_at[0], np.sin(2.5), atol=1e-3)

# Edge case: a very large length-scale makes the kernel nearly constant, so std
# stays small and roughly uniform everywhere instead of blowing up far from data.
gp5 = GaussianProcessRegression(kernel='rbf', kernel_params={'sigma': 1.0, 'length_scale': 1e4}, noise=1e-6)
gp5.fit(X5, np.sin(X5).ravel())
_, std_near_large = gp5.predict(np.array([[2.5]]), return_std=True)
_, std_far_large  = gp5.predict(np.array([[20.0]]), return_std=True)
assert std_near_large[0] < 0.01 and std_far_large[0] < 0.01   # both tiny...
assert std_far_large[0] < 5 * std_near_large[0]               # ...and roughly the same order of magnitude

print('✓ GaussianProcessRegression matches DML 186 and handles the single-point / at-train-point / large-length-scale edge cases')

<details>
<summary>Solution</summary>

```python
def rbf_kernel_dml(x, x_prime, sigma=1.0, length_scale=1.0):
    sq_norm = np.linalg.norm(x - x_prime) ** 2
    return sigma**2 * np.exp(-sq_norm / (2 * length_scale**2))

def linear_kernel_dml(x, x_prime, sigma_b=1.0, sigma_v=1.0):
    return sigma_b**2 + sigma_v**2 * np.dot(x, x_prime)

class GaussianProcessRegression:
    def __init__(self, kernel='rbf', noise=1e-5, kernel_params=None):
        self.kernel_name = kernel
        self.noise = noise
        self.kernel_params = kernel_params if kernel_params else {}

    def _select_kernel(self, x1, x2):
        if self.kernel_name == 'rbf':
            return rbf_kernel_dml(x1, x2, **self.kernel_params)
        elif self.kernel_name == 'linear':
            return linear_kernel_dml(x1, x2, **self.kernel_params)
        raise ValueError(f"Unsupported kernel: {self.kernel_name}")

    def _compute_covariance(self, X1, X2):
        X1, X2 = np.atleast_2d(X1), np.atleast_2d(X2)
        K = np.zeros((len(X1), len(X2)))
        for i in range(len(X1)):
            for j in range(len(X2)):
                K[i, j] = self._select_kernel(X1[i], X2[j])
        return K

    def fit(self, X, y):
        self.X_train = np.asarray(X, dtype=float)
        self.y_train = np.asarray(y, dtype=float)
        self.K_ = self._compute_covariance(self.X_train, self.X_train) + self.noise * np.eye(len(self.X_train))
        self.alpha_ = np.linalg.solve(self.K_, self.y_train)

    def predict(self, X_test, return_std=False):
        X_test = np.atleast_2d(X_test)
        K_s = self._compute_covariance(self.X_train, X_test)
        mu = K_s.T @ self.alpha_
        if return_std:
            K_ss = self._compute_covariance(X_test, X_test)
            cov = K_ss - K_s.T @ np.linalg.solve(self.K_, K_s)
            return mu, np.sqrt(np.clip(np.diag(cov), 0, None))
        return mu
```

`np.linalg.solve` plays the same role as the reference solution's Cholesky
decomposition + triangular solves -- both compute `K⁻¹y` and `K⁻¹K_s` without
forming an explicit inverse; `solve` is just less numerically specialized
(fine at the scale of these exercises).

The single-training-point and at-a-training-point edge cases land at (nearly)
`sqrt(noise)` for the same reason as the `posterior_var` exercise above: with
one dominant nearby observation, the predictive variance reduces to
`noise · σ² / (σ² + noise) ≈ noise`. The huge-length-scale case shows the
failure mode of an over-smooth kernel: std stays tiny *everywhere*, including
far outside the training range -- confident, but for the wrong reason, since
the kernel no longer distinguishes "near the data" from "far from it."

</details>